In [1]:
import pandas as pd

# 1. LOAD RAW DATA

df = pd.read_csv("Airline Dataset.csv")
print("Initial shape:", df.shape)

# 2. REMOVE DUPLICATES

df.drop_duplicates(inplace=True)

# 3. DROP COLUMNS

df.drop(columns=[
    "Airport Country Code",
    "Airport Continent",
    "Arrival Airport",   # incorrect column
    "First Name",
    "Last Name",
    "Pilot Name"
], inplace=True)


# 4. STANDARDISE TEXT

df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

df["Country Name"] = df["Country Name"].str.title()
df["Nationality"] = df["Nationality"].str.title()
df["Flight Status"] = df["Flight Status"].str.title()


# 5. REMOVE INVALID AGE VALUES

df = df[(df["Age"] > 0) & (df["Age"] < 100)]

# 6. CREATE AGE GROUP

def age_group(age):
    if age < 18:
        return "Young"
    elif age < 60:
        return "Adult"
    else:
        return "Senior"

df["Age_Group"] = df["Age"].apply(age_group)


# 7. DATE TRANSFORMATIONS 
df["Departure Date"] = pd.to_datetime(df["Departure Date"]).dt.date

df["Day"] = pd.to_datetime(df["Departure Date"]).dt.day
df["Month"] = pd.to_datetime(df["Departure Date"]).dt.month
df["Month_Name"] = pd.to_datetime(df["Departure Date"]).dt.month_name()
df["Year"] = pd.to_datetime(df["Departure Date"]).dt.year
df["Day_of_Week"] = pd.to_datetime(df["Departure Date"]).dt.day_name()
df["Quarter"] = pd.to_datetime(df["Departure Date"]).dt.quarter

df["is_weekend"] = df["Day_of_Week"].isin(["Saturday", "Sunday"]).astype(int)


# 8.FACT MEASURES

df["delay_flag"] = (df["Flight Status"] == "Delayed").astype(int)
df["cancel_flag"] = (df["Flight Status"] == "Cancelled").astype(int)
df["flight_count"] = 1



df.to_csv("Airline_ETL_final.csv", index=False)

# DIMENSIONS

# DIM DATE 
dim_date = df[[
    "Departure Date", "Day", "Month", "Month_Name",
    "Year", "Day_of_Week", "Quarter", "is_weekend"
]].drop_duplicates().reset_index(drop=True)

dim_date.columns = [
    "full_date", "day", "month", "month_name",
    "year", "day_of_week", "quarter", "is_weekend"
]

dim_date.insert(0, "date_id", range(1, len(dim_date) + 1))

# DIM LOCATION
dim_location = df[[
    "Airport Name", "Country Name", "Continents"
]].drop_duplicates().reset_index(drop=True)

dim_location.columns = ["airport_name", "country_name", "continent"]
dim_location.insert(0, "location_id", range(1, len(dim_location) + 1))

# DIM PASSENGER 
dim_passenger = df[[
    "Passenger ID", "Gender", "Age", "Age_Group", "Nationality"
]].drop_duplicates().reset_index(drop=True)

dim_passenger.columns = [
    "passenger_id", "gender", "age", "age_group", "nationality"
]

dim_passenger.insert(0, "passenger_key", range(1, len(dim_passenger) + 1))

# -------- DIM FLIGHT --------
dim_flight = df[["Flight Status"]].drop_duplicates().reset_index(drop=True)

dim_flight.columns = ["flight_status"]
dim_flight.insert(0, "flight_id", range(1, len(dim_flight) + 1))

# FACT TABLE 

fact = df.copy()

fact = fact.merge(
    dim_date,
    left_on=["Departure Date", "Day", "Month", "Month_Name",
             "Year", "Day_of_Week", "Quarter", "is_weekend"],
    right_on=["full_date", "day", "month", "month_name",
              "year", "day_of_week", "quarter", "is_weekend"],
    how="left"
)

fact = fact.merge(
    dim_location,
    left_on=["Airport Name", "Country Name", "Continents"],
    right_on=["airport_name", "country_name", "continent"],
    how="left"
)


fact = fact.merge(
    dim_passenger,
    left_on=["Passenger ID", "Gender", "Age", "Age_Group", "Nationality"],
    right_on=["passenger_id", "gender", "age", "age_group", "nationality"],
    how="left"
)


fact = fact.merge(
    dim_flight,
    left_on="Flight Status",
    right_on="flight_status",
    how="left"
)

# FACT TABLE
fact_flight = fact[[
    "date_id",
    "location_id",
    "passenger_key",
    "flight_id",
    "flight_count",
    "delay_flag",
    "cancel_flag"
]].copy()

fact_flight.insert(0, "fact_id", range(1, len(fact_flight) + 1))

dim_date.to_csv("dim_date.csv", index=False)
dim_location.to_csv("dim_location.csv", index=False)
dim_passenger.to_csv("dim_passenger.csv", index=False)
dim_flight.to_csv("dim_flight.csv", index=False)
fact_flight.to_csv("fact_flight.csv", index=False)

Files created:
- Airline_ETL.csv
- dim_date.csv
- dim_location.csv
- dim_passenger.csv
- dim_flight.csv
- fact_flight.csv


dim_date.csv updated to YYYY-MM-DD format ✅


C:\Users\dhruv\AppData\Local\Temp\ipykernel_18948\2260769879.py:7: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dim_date["full_date"] = pd.to_datetime(dim_date["full_date"]).dt.date
